# Project 4 — Notebook 2: Query Demonstration

Game of Thrones Lore RAG System

This notebook demonstrates queries against the `got_lore` Elasticsearch index.

## Imports

In [15]:
import os
from elasticsearch import Elasticsearch, helpers
import urllib3
from pprint import pprint

# Disable the specific InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Connect to Elasticsearch

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

ES_USER = os.getenv('ES_USER', 'elastic')
ES_PASSWORD = os.environ['ES_PASSWORD']  # set in .env
ES_HOST = os.getenv('ES_HOST', 'https://localhost:9200/')

client = Elasticsearch(
    ES_HOST,
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False,
)

In [19]:
client.info()

ObjectApiResponse({'name': 'f08d4b694cd1', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'sHThI9v4QFK889Tu6Z4A8w', 'version': {'number': '9.3.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '0dd66e52ba3aa076cf498264e46339dbb71f0269', 'build_date': '2026-02-23T23:37:38.684779921Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Query 1: Document Count

[1] Count the total number of documents in the index. There must be at least 10,000 sentences.

In [20]:
client.cat.count(index='got_lore')

TextApiResponse('1777499781 21:56:21 24103\n')

## Query 2: Keyword Query on Metadata

[2] Retrieve all sentences extracted from the article about House Stark by querying the `doc_title` keyword field.

In [21]:
query = {'query': {'term': {'doc_title': 'House Stark'}}}

results_obj = client.search(index='got_lore', body=query)
results = dict(results_obj)

print(f"Returned {results['hits']['total']['value']} documents")
pprint(results['hits']['hits'][0])

Returned 1013 documents
{'_id': '10197',
 '_index': 'got_lore',
 '_score': 3.1689682,
 '_source': {'category': 'house',
             'doc_title': 'House Stark',
             'sentence': "George R. R. Martin's A Song of Ice and Fire saga "
                         'features a large cast of characters.',
             'sentence_id': 10197}}


[3] We can also filter by the `category` keyword field to get all sentences from character articles.

In [22]:
query = {'query': {'term': {'category': 'character'}}}

results_obj = client.search(index='got_lore', body=query)
results = dict(results_obj)

print(f"Returned {results['hits']['total']['value']} documents")
pprint(results['hits']['hits'][0])

Returned 8796 documents
{'_id': '1401',
 '_index': 'got_lore',
 '_score': 1.0080239,
 '_source': {'category': 'character',
             'doc_title': 'Jon Snow (character)',
             'sentence': 'Jon Snow is a fictional main character in the A Song '
                         'of Ice and Fire series of epic fantasy novels by '
                         'American author George R. R. Martin, and its HBO '
                         'television adaptation Game of Thrones, in which he '
                         'is portrayed by Kit Harington.',
             'sentence_id': 1401}}


## Query 3: Match Query on Sentence Text

[4] Search for sentences mentioning dragons.

In [23]:
query = {'query': {'match': {'sentence': 'dragon fire breath'}}}

results_obj = client.search(index='got_lore', body=query)
results = dict(results_obj)

print(f"Returned {results['hits']['total']['value']} documents")
pprint(results['hits']['hits'][0])

Returned 1419 documents
{'_id': '6829',
 '_index': 'got_lore',
 '_score': 14.993935,
 '_source': {'category': 'character',
             'doc_title': 'Euron Greyjoy',
             'sentence': 'Its coat of arms shows a red, three-headed, two '
                         'legged dragon breathing fire on a black field, and '
                         'its words are Fire and Blood.',
             'sentence_id': 6829}}


[5] Search for sentences about betrayal and death.

In [24]:
query = {'query': {'match': {'sentence': 'betrayal murder death assassination'}}}

results_obj = client.search(index='got_lore', body=query)
results = dict(results_obj)

print(f"Returned {results['hits']['total']['value']} documents")
pprint(results['hits']['hits'][0])

Returned 1510 documents
{'_id': '1619',
 '_index': 'got_lore',
 '_score': 14.087414,
 '_source': {'category': 'character',
             'doc_title': 'Jon Snow (character)',
             'sentence': 'They assassinate Jon for his perceived betrayal of '
                         "the Night's Watch.",
             'sentence_id': 1619}}


## Query 4: KNN Query

[6] Run a KNN query using an embedding of the query text to find semantically similar sentences.

In [25]:
query_text = 'Who are the heirs to the Iron Throne?'
query_vector = model.encode(query_text)

query = {
    'query': {
        'knn': {
            'field': 'embedding',
            'query_vector': query_vector,
            'num_candidates': 20
        }
    }
}

results_obj = client.search(index='got_lore', body=query)
results = dict(results_obj)

print(f"Returned {results['hits']['total']['value']} documents")
pprint(results['hits']['hits'][0])

Returned 10 documents
{'_id': '4935',
 '_index': 'got_lore',
 '_score': 0.8548844,
 '_source': {'category': 'character',
             'doc_title': 'Yara Greyjoy',
             'sentence': "He is the eldest of Cersei Lannister's children and "
                         'heir to the Iron Throne.',
             'sentence_id': 4935}}


[7] Print a more readable summary of the top KNN results.

In [26]:
for i, hit in enumerate(results['hits']['hits']):
    print(f"Result {i}  Score {hit['_score']:.4f}  ({hit['_source']['doc_title']})")
    print(f"  {hit['_source']['sentence'][:120]}")

Result 0  Score 0.8549  (Yara Greyjoy)
  He is the eldest of Cersei Lannister's children and heir to the Iron Throne.
Result 1  Score 0.8548  (Direwolf (Game of Thrones))
  He is the eldest of Cersei Lannister's children and heir to the Iron Throne.
Result 2  Score 0.8503  (Game of Thrones season 4)
  After the death of Robb Stark at The Red Wedding, all three remaining kings in Westeros believe they have a claim to the
Result 3  Score 0.8391  (Daenerys Targaryen)
  She becomes the heir of the Targaryen dynasty after her brother's murder and plans to reclaim the Iron Throne herself, s
Result 4  Score 0.8287  (Direwolf (Game of Thrones))
  Since Robert's family had closer ties to the former Royal family, he was put on the Iron Throne.
Result 5  Score 0.8257  (Cersei Lannister)
  In the power vacuum following Tommen's death, Cersei claims the Iron Throne as the first queen regnant of the Seven King
Result 6  Score 0.8227  (Euron Greyjoy)
  The Greyjoys became Lords Paramount of the Iron 

## Query 5: Hybrid Query combining Match and KNN

[8] Combine a match query and a KNN query using a `bool` query with `should` clauses. Documents that score well on both term matching and semantic similarity will rank highest.

In [27]:
query_text = 'Who are the heirs to the Iron Throne?'
query_vector = model.encode(query_text)

query = {'query': {
    'bool': {
        'should': [
            {'match': {'sentence': query_text}},
            {'knn': {
                'field': 'embedding',
                'query_vector': query_vector,
                'k': 10,
                'num_candidates': 50
            }}
        ]
    }
},
'size': 10
}

results_obj = client.search(index='got_lore', body=query)
results = dict(results_obj)

print(f"Returned {results['hits']['total']['value']} documents")
pprint(results['hits']['hits'][0])

Returned 10000 documents
{'_id': '4007',
 '_index': 'got_lore',
 '_score': 15.210882,
 '_source': {'category': 'character',
             'doc_title': 'Stannis Baratheon',
             'sentence': "After Robert's death, Stannis claims himself the "
                         "true heir to the Iron Throne as Cersei's children "
                         'are bastards born of incest.',
             'sentence_id': 4007}}


[9] Print a readable summary of the hybrid results.

In [28]:
for i, hit in enumerate(results['hits']['hits']):
    print(f"Result {i}  Score {hit['_score']:.4f}  ({hit['_source']['doc_title']})")
    print(f"  {hit['_source']['sentence'][:120]}")

Result 0  Score 15.2109  (Stannis Baratheon)
  After Robert's death, Stannis claims himself the true heir to the Iron Throne as Cersei's children are bastards born of 
Result 1  Score 15.1596  (Yara Greyjoy)
  He is the eldest of Cersei Lannister's children and heir to the Iron Throne.
Result 2  Score 15.1595  (Direwolf (Game of Thrones))
  He is the eldest of Cersei Lannister's children and heir to the Iron Throne.
Result 3  Score 13.4406  (Game of Thrones season 7)
  Bran learns that Jon is really his cousin, Aegon Targaryen, the legitimate heir to the Iron Throne.
Result 4  Score 12.5546  (Daenerys Targaryen)
  She becomes the heir of the Targaryen dynasty after her brother's murder and plans to reclaim the Iron Throne herself, s
Result 5  Score 12.2590  (Joffrey Baratheon)
  Family tree
TV adaptation
Season 1
In Season 1 of Game of Thrones, Joffrey Baratheon is introduced as the arrogant and c
Result 6  Score 12.2112  (Westeros)
  Aegon's progeny reigned as kings of the Seven Kingd